# Day 3 — Program Logic Flowcharts

Flowcharts for every function in:
- **Program 1 — Age Verification System**
- **Program 2 — Grade Calculator**

### Shape legend

| Shape | Colour | Means |
|---|---|---|
| Rounded rectangle | 🟢 Green | `START` / `END` terminal, or `return` |
| Rectangle | 🔵 Blue | Process / computation |
| Parallelogram | 🟣 Purple | Input / Output (`input()` / `print()`) |
| Diamond | 🟠 Orange-red | Decision (branch point) |

### Arrow conventions
- **Bottom** of a diamond = main flow continues (the *not-taken* branch continues downward)
- **Right** of a diamond = the labelled branch departs
- **Dashed vertical line** on the right = shared return highway; all error branches rejoin here and re-enter the input node


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np

# ── Palette ───────────────────────────────────────────────────────────────────
BG     = "#1E1E2E"
C_TERM = "#2E7D32"   # terminal  – green
C_PROC = "#1565C0"   # process   – dark blue
C_IO   = "#6A1B9A"   # I/O       – deep purple
C_DEC  = "#BF360C"   # decision  – burnt orange
C_ARR  = "#90A4AE"   # arrows    – cool grey
C_LBL  = "#FFD54F"   # labels    – amber
C_NOTE = "#546E7A"   # annotations

# ── Shape helpers ─────────────────────────────────────────────────────────────
def _txt(ax, cx, cy, t, fs=8.5, bold=False):
    ax.text(cx, cy, t, ha="center", va="center", fontsize=fs,
            fontweight="bold" if bold else "normal",
            color="white", zorder=4, multialignment="center")

def node_term(ax, cx, cy, txt, w=2.8, h=0.54):
    ax.add_patch(FancyBboxPatch((cx-w/2, cy-h/2), w, h,
        boxstyle="round,pad=0.14", fc=C_TERM, ec="white", lw=1.5, zorder=3))
    _txt(ax, cx, cy, txt, 9.5, True)
    return {"T":(cx,cy+h/2),"B":(cx,cy-h/2),"L":(cx-w/2,cy),"R":(cx+w/2,cy)}

def node_proc(ax, cx, cy, txt, w=3.8, h=0.54):
    ax.add_patch(FancyBboxPatch((cx-w/2, cy-h/2), w, h,
        boxstyle="square,pad=0.05", fc=C_PROC, ec="white", lw=1.5, zorder=3))
    _txt(ax, cx, cy, txt)
    return {"T":(cx,cy+h/2),"B":(cx,cy-h/2),"L":(cx-w/2,cy),"R":(cx+w/2,cy)}

def node_io(ax, cx, cy, txt, w=4.0, h=0.54):
    sk = 0.22
    pts = np.array([[cx-w/2+sk,cy+h/2],[cx+w/2+sk,cy+h/2],
                    [cx+w/2-sk,cy-h/2],[cx-w/2-sk,cy-h/2]])
    ax.add_patch(plt.Polygon(pts, fc=C_IO, ec="white", lw=1.5, zorder=3))
    _txt(ax, cx, cy, txt)
    return {"T":(cx,cy+h/2),"B":(cx,cy-h/2),"L":(cx-w/2,cy),"R":(cx+w/2,cy)}

def node_dec(ax, cx, cy, txt, hw=1.5, hh=0.85):
    pts = np.array([[cx,cy+hh],[cx+hw,cy],[cx,cy-hh],[cx-hw,cy]])
    ax.add_patch(plt.Polygon(pts, fc=C_DEC, ec="white", lw=1.5, zorder=3))
    _txt(ax, cx, cy, txt, 8)
    return {"T":(cx,cy+hh),"B":(cx,cy-hh),"L":(cx-hw,cy),"R":(cx+hw,cy)}

# ── Arrow helpers ─────────────────────────────────────────────────────────────
def arr(ax, p1, p2, lbl="", lside="r"):
    ax.annotate("", xy=p2, xytext=p1,
        arrowprops=dict(arrowstyle="->", color=C_ARR, lw=1.8,
                        connectionstyle="arc3,rad=0"), zorder=2)
    if lbl:
        mx, my = (p1[0]+p2[0])/2, (p1[1]+p2[1])/2
        ox = 0.32 if lside == "r" else -0.32
        ax.text(mx+ox, my, lbl, fontsize=8, color=C_LBL, ha="center", va="center",
                bbox=dict(boxstyle="round,pad=0.15", fc="#2A2A3E", ec="none", alpha=0.9),
                zorder=5)

def loop_left(ax, from_pt, to_pt, via_x):
    """L-path: from_pt → via_x (horiz) → to_pt.y (vert) → to_pt (arrow)."""
    fx, fy = from_pt;  tx, ty = to_pt
    ax.plot([fx, via_x], [fy, fy], color=C_ARR, lw=1.8, zorder=2)
    ax.plot([via_x, via_x], [fy, ty], color=C_ARR, lw=1.8, zorder=2)
    ax.annotate("", xy=(tx, ty), xytext=(via_x, ty),
        arrowprops=dict(arrowstyle="->", color=C_ARR, lw=1.8), zorder=2)

def highway_right(ax, nodes_y, hx, input_r):
    """Shared right-side return highway.  nodes_y = [(node_dict, cy), ...]"""
    min_y = min(cy for _, cy in nodes_y)
    max_y = input_r[1]
    for n, cy in nodes_y:
        ax.plot([n["R"][0], hx], [cy, cy], color=C_ARR, lw=1.5, zorder=2)
    ax.plot([hx, hx], [min_y-0.3, max_y], color=C_ARR, lw=2.0, ls="--", zorder=2)
    ax.annotate("", xy=input_r, xytext=(hx, max_y),
        arrowprops=dict(arrowstyle="->", color=C_ARR, lw=1.8), zorder=2)
    ax.text(hx+0.45, (min_y+max_y)/2, "↺\nretry", color=C_NOTE, fontsize=8.5,
            style="italic", rotation=90, ha="center", va="center")

def new_fig(w, h, title):
    fig, ax = plt.subplots(figsize=(w, h))
    fig.patch.set_facecolor(BG);  ax.set_facecolor(BG);  ax.axis("off")
    ax.set_title(title, color="white", fontsize=12, fontweight="bold", pad=14)
    return fig, ax

print("✓  Drawing helpers loaded.")


---
## Program 1 — Age Verification System

### `get_name()` — Input validation loop
Loops until the user enters a non-empty, non-whitespace-only name.  
`str.strip()` is called first, so `"   "` (spaces only) is still rejected.


In [ ]:
fig, ax = new_fig(8, 8, "get_name()  —  Input Validation Loop")
ax.set_xlim(-5, 5.5);  ax.set_ylim(3, 12)

s  = node_term(ax,  0, 11,   "START")
ni = node_io  (ax,  0,  9.5, "name = input('Enter name: ').strip()", w=4.2)
d  = node_dec (ax,  0,  7.7, "name\ntruthy?", hw=1.5, hh=0.85)
nr = node_term(ax,  3.8, 7.7, "return name")
pe = node_io  (ax,  0,  5.7, "print('Name cannot be empty')", w=3.8)

arr(ax, s["B"],  ni["T"])
arr(ax, ni["B"], d["T"])
arr(ax, d["R"],  nr["L"], "Yes")
arr(ax, d["B"],  pe["T"], "No")
loop_left(ax, pe["B"], ni["L"], -3.4)

ax.text(-4.0, 7.7, "↺ retry", color=C_NOTE, fontsize=8, style="italic")
plt.tight_layout();  plt.show()


### `get_age()` — Numeric validation loop
Three distinct failure modes, each jumping back to the same input node:
1. `int(raw)` raises `ValueError` → caught by `except`, `continue` fires
2. `age < 0` → negative ages are nonsensical
3. `age > 130` → above the sanity ceiling (`MAX_AGE`)

All three failure paths share the **dashed right-side highway** and re-enter the input node.


In [ ]:
fig, ax = new_fig(11, 14, "get_age()  —  Numeric Validation Loop")
ax.set_xlim(-4, 9);  ax.set_ylim(1.5, 16)

s   = node_term(ax,  0, 15,   "START")
ni  = node_io  (ax,  0, 13.3, "raw = input('Enter age: ').strip()", w=4.4)
d1  = node_dec (ax,  0, 11.4, "int(raw)\nsucceeds?",    hw=1.65, hh=0.92)
pe1 = node_io  (ax,  4.8, 11.4, "print('Not a\nvalid number')", w=3.0)
p1  = node_proc(ax,  0,  9.5,  "age = int(raw)",          w=3.2)
d2  = node_dec (ax,  0,  7.7,  "age < 0?",                hw=1.45, hh=0.82)
pe2 = node_io  (ax,  4.8,  7.7, "print('Age cannot\nbe negative')", w=3.0)
d3  = node_dec (ax,  0,  5.5,  "age > 130?",              hw=1.45, hh=0.82)
pe3 = node_io  (ax,  4.8,  5.5, "print('Age above\n130? Unlikely')", w=3.0)
ret = node_term(ax,  0,  3.7,  "return age")
e   = node_term(ax,  0,  2.4,  "END")

# Main success path
arr(ax, s["B"],   ni["T"])
arr(ax, ni["B"],  d1["T"])
arr(ax, d1["B"],  p1["T"],  "Yes")
arr(ax, p1["B"],  d2["T"])
arr(ax, d2["B"],  d3["T"],  "No")
arr(ax, d3["B"],  ret["T"], "No")
arr(ax, ret["B"], e["T"])

# Fail branches → right
arr(ax, d1["R"],  pe1["L"], "No")
arr(ax, d2["R"],  pe2["L"], "Yes")
arr(ax, d3["R"],  pe3["L"], "Yes")

# Shared return highway (right side, x=7.2)
highway_right(ax, [(pe1,11.4),(pe2,7.7),(pe3,5.5)], 7.2, ni["R"])

plt.tight_layout();  plt.show()


### `classify_age()` — `if / elif / elif / else` chain
Each branch returns immediately — Python never reaches the next condition once a match fires.  
The annotations on the left show the **implicit lower bound** carried by each `elif`  
(e.g. `age ≤ 17?` only runs if `age ≤ 12?` was already `False`, so `age > 12` is guaranteed).


In [ ]:
fig, ax = new_fig(9.5, 11, "classify_age()  —  if / elif / elif / else Chain")
ax.set_xlim(-5, 6.5);  ax.set_ylim(2.5, 13)

s  = node_term(ax,  0, 12,   "START: (user_name, age)")
d1 = node_dec (ax,  0, 10.4, "age ≤ 12?",   hw=1.4, hh=0.82)
r1 = node_term(ax,  4.2, 10.4, "return\n\"child\" greeting", w=3.0)
d2 = node_dec (ax,  0,  8.4,  "age ≤ 17?",  hw=1.4, hh=0.82)
r2 = node_term(ax,  4.2,  8.4, "return\n\"teenager\" greeting", w=3.0)
d3 = node_dec (ax,  0,  6.4,  "age ≤ 64?",  hw=1.4, hh=0.82)
r3 = node_term(ax,  4.2,  6.4, "return\n\"adult\" greeting", w=3.0)
r4 = node_term(ax,  0,  4.5,  "return\n\"senior\" greeting", w=3.2)

arr(ax, s["B"],  d1["T"])
arr(ax, d1["R"], r1["L"], "Yes")
arr(ax, d1["B"], d2["T"], "No")
arr(ax, d2["R"], r2["L"], "Yes")
arr(ax, d2["B"], d3["T"], "No")
arr(ax, d3["R"], r3["L"], "Yes")
arr(ax, d3["B"], r4["T"], "No\n(else)")

# Implied lower-bound annotations
notes = [
    (9.4,  "age > 12 implied\n(if failed above)"),
    (7.4,  "age > 17 implied\n(elif failed above)"),
    (5.4,  "age > 64 implied\n(elif failed above)"),
]
for y, txt in notes:
    ax.text(-4.0, y, txt, color=C_NOTE, fontsize=7.5, style="italic",
            ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.2", fc="#2A2A3E", ec="none", alpha=0.7))

plt.tight_layout();  plt.show()


### `main` — Age Verification System
Top-level flow: three sequential function calls, one `print`.  
The `if __name__ == "__main__":` guard means this only runs when the file is executed directly.


In [ ]:
fig, ax = new_fig(8, 9, "Age Verification System  —  Main Flow  (__main__)")
ax.set_xlim(-4, 5);  ax.set_ylim(2, 13)

s  = node_term(ax, 0, 12,   "START")
g  = node_proc(ax, 0, 10.4, 'user_name = get_name()', w=4.2)
h  = node_proc(ax, 0,  8.8, 'user_age  = get_age()',  w=4.2)
c  = node_proc(ax, 0,  7.2, 'result = classify_age(\nuser_name, user_age)', w=4.2)
p  = node_io  (ax, 0,  5.4, 'print(result)',           w=3.6)
e  = node_term(ax, 0,  3.8, "END")

arr(ax, s["B"], g["T"])
arr(ax, g["B"], h["T"])
arr(ax, h["B"], c["T"])
arr(ax, c["B"], p["T"])
arr(ax, p["B"], e["T"])

for n, label in [(g,"→ see get_name() chart"),(h,"→ see get_age() chart"),
                 (c,"→ see classify_age() chart")]:
    ax.text(n["R"][0]+0.15, n["R"][1], label,
            color=C_NOTE, fontsize=7.5, style="italic", va="center")

plt.tight_layout();  plt.show()


---
## Program 2 — Grade Calculator

### `get_grade()` — Numeric validation loop
Same structure as `get_age()` but simpler: one `try/except` and one range check.  
`float()` is used (not `int()`) to accept decimal grades like `85.5`.


In [ ]:
fig, ax = new_fig(10, 12, "get_grade()  —  Numeric Validation Loop")
ax.set_xlim(-4, 8);  ax.set_ylim(2, 15)

s   = node_term(ax,  0, 14,   "START")
ni  = node_io  (ax,  0, 12.4, "raw = input('Enter grade (0-100): ').strip()", w=5.0)
d1  = node_dec (ax,  0, 10.5, "float(raw)\nsucceeds?",  hw=1.65, hh=0.92)
pe1 = node_io  (ax,  4.8, 10.5, "print('Not a number')",  w=2.8)
p1  = node_proc(ax,  0,  8.7, "grade = float(raw)",       w=3.4)
d2  = node_dec (ax,  0,  6.9, "0 ≤ grade ≤ 100?",        hw=1.75, hh=0.88)
pe2 = node_io  (ax,  4.8,  6.9, "print('Must be\n0 – 100')", w=2.8)
ret = node_term(ax,  0,  5.0, "return grade")
e   = node_term(ax,  0,  3.7, "END")

arr(ax, s["B"],   ni["T"])
arr(ax, ni["B"],  d1["T"])
arr(ax, d1["B"],  p1["T"],  "Yes")
arr(ax, p1["B"],  d2["T"])
arr(ax, d2["B"],  ret["T"], "Yes")
arr(ax, ret["B"], e["T"])
arr(ax, d1["R"],  pe1["L"], "No")
arr(ax, d2["R"],  pe2["L"], "No")

highway_right(ax, [(pe1,10.5),(pe2,6.9)], 7.0, ni["R"])

plt.tight_layout();  plt.show()


### `classify_grade()` — `if / elif / elif / elif / else` chain
Five possible returns, ordered **high to low**.  
Each `elif` implicitly inherits the lower bound from all failed conditions above it —  
`grade ≥ 80?` only runs when `grade ≥ 90?` was already `False`, so `grade < 90` is guaranteed.


In [ ]:
fig, ax = new_fig(10, 13, "classify_grade()  —  if / elif / elif / elif / else Chain")
ax.set_xlim(-5, 7);  ax.set_ylim(1.5, 14)

s  = node_term(ax,  0, 13,   "START: grade ∈ [0, 100]")
d1 = node_dec (ax,  0, 11.5, "grade ≥ 90?", hw=1.55, hh=0.82)
r1 = node_term(ax,  4.6, 11.5, "return A\n\"Excellent work!\"",          w=3.0)
d2 = node_dec (ax,  0,  9.5,  "grade ≥ 80?", hw=1.55, hh=0.82)
r2 = node_term(ax,  4.6,  9.5, "return B\n\"Good job!\"",                w=3.0)
d3 = node_dec (ax,  0,  7.5,  "grade ≥ 70?", hw=1.55, hh=0.82)
r3 = node_term(ax,  4.6,  7.5, "return C\n\"Fair effort.\"",             w=3.0)
d4 = node_dec (ax,  0,  5.5,  "grade ≥ 60?", hw=1.55, hh=0.82)
r4 = node_term(ax,  4.6,  5.5, "return D\n\"You passed, barely.\"",      w=3.0)
r5 = node_term(ax,  0,   3.7,  "return F\n\"Failed this time.\"",         w=3.4)

arr(ax, s["B"],  d1["T"])
arr(ax, d1["R"], r1["L"], "Yes")
arr(ax, d1["B"], d2["T"], "No")
arr(ax, d2["R"], r2["L"], "Yes")
arr(ax, d2["B"], d3["T"], "No")
arr(ax, d3["R"], r3["L"], "Yes")
arr(ax, d3["B"], d4["T"], "No")
arr(ax, d4["R"], r4["L"], "Yes")
arr(ax, d4["B"], r5["T"], "No (else)")

notes = [
    (10.5, "80 ≤ grade < 90\nimplied"),
    ( 8.5, "70 ≤ grade < 80\nimplied"),
    ( 6.5, "60 ≤ grade < 70\nimplied"),
]
for y, txt in notes:
    ax.text(-4.0, y, txt, color=C_NOTE, fontsize=7.5, style="italic",
            ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.2", fc="#2A2A3E", ec="none", alpha=0.7))

plt.tight_layout();  plt.show()


### `main` — Grade Calculator
Note the **tuple unpack** on the `classify_grade()` call:  
`letter, message, interp = classify_grade(grade)` — one call, three return values.


In [ ]:
fig, ax = new_fig(8.5, 9.5, "Grade Calculator  —  Main Flow  (__main__)")
ax.set_xlim(-4.5, 5);  ax.set_ylim(2, 13)

s  = node_term(ax, 0, 12,   "START")
f1 = node_proc(ax, 0, 10.4, "grade = get_grade()",              w=4.4)
f2 = node_proc(ax, 0,  8.7, "letter, message, interp =\nclassify_grade(grade)", w=4.4)
p1 = node_io  (ax, 0,  6.9, 'print(f"Grade: {grade}  →  {letter}")', w=4.6)
p2 = node_io  (ax, 0,  5.2, 'print(f"{message} {interp}")',    w=4.6)
e  = node_term(ax, 0,  3.7, "END")

arr(ax, s["B"],  f1["T"])
arr(ax, f1["B"], f2["T"])
arr(ax, f2["B"], p1["T"])
arr(ax, p1["B"], p2["T"])
arr(ax, p2["B"], e["T"])

for n, lbl in [(f1,"→ see get_grade() chart"),
               (f2,"→ see classify_grade() chart")]:
    ax.text(n["R"][0]+0.15, n["R"][1], lbl,
            color=C_NOTE, fontsize=7.5, style="italic", va="center")

plt.tight_layout();  plt.show()
